# 01 — Data Preparation

## Overview

This notebook rebuilds four simulated client extracts from the unchanged source copy, standardises their dates and fields, and exports one continuous 209-week modelling table. Original media blanks are flagged before zero-filling, competitor-spend gaps are interpolated, and missing sales remain missing for transparent downstream handling.

## Context & Methods

The source is split into sales, digital media, offline media, and market-context extracts to demonstrate a refreshable multi-file preparation workflow. Reusable functions in `src/` perform validation, weekly alignment, aggregation, treatment logging, and the final one-to-one merge.

### Key Assumptions

- A blank media value is treated as an inactive campaign week only for this synthetic portfolio dataset; the original blank is preserved in a `_was_missing` flag.
- Competitor-spend gaps are interpolated in weekly order, with boundary filling only if needed.
- Missing sales are retained for reporting and excluded later from model fitting and scoring.
- The downloaded CSV is never overwritten. It is copied into the project only when the project source copy is absent.

### 1. Set Up the Project Path

In [1]:
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
PROJECT_ROOT = (
    working_directory.parent
    if working_directory.name.lower() == "notebooks"
    else working_directory
)
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError(
        "Launch this notebook from the project root or the notebooks directory."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\yifei\Workspace\marketing-mmm-python


In [4]:
import shutil

import pandas as pd
from IPython.display import display

from src.config import get_project_paths
from src.data_prep import (
    prepare_modelling_table,
    standardize_columns,
    to_monday_week_start,
)
from src.sample_extracts import create_client_extracts

paths = get_project_paths(PROJECT_ROOT)
for folder in (
    paths.source,
    paths.client_extracts,
    paths.processed,
    paths.reports,
):
    folder.mkdir(parents=True, exist_ok=True)

## Data

### 2. Preserve the Source and Create Client Extracts

In [ ]:
download_source = Path("C:/Users/yanyi/Downloads/mmm_dataset.csv")
project_source = paths.source / "mmm_dataset.csv"

if project_source.exists():
    source_action = "Reused the existing unchanged project source copy."
else:
    if not download_source.exists():
        raise FileNotFoundError(
            f"Source CSV is missing from both project and download paths: {download_source}"
        )
    shutil.copy2(download_source, project_source)
    source_action = "Copied the download into data/raw/source without altering it."

extract_paths = create_client_extracts(project_source, paths.client_extracts)
print(source_action)
print(f"Created {len(extract_paths)} client extracts.")

### 3. Preview Bounded Source Samples

source_preview = pd.read_csv(project_source).head(5)
display(source_preview)

sales_preview = standardize_columns(
    pd.read_csv(extract_paths["sales"]).head(3)
)
sales_preview.insert(
    0,
    "canonical_monday",
    to_monday_week_start(sales_preview["financial_week"]),
)
print("Standardised sales columns with canonical Monday dates")
display(sales_preview)

for extract_name, extract_path in extract_paths.items():
    if extract_path.suffix.lower() == ".xlsx":
        preview = pd.read_excel(extract_path).head(3)
    else:
        preview = pd.read_csv(extract_path).head(3)
    print(f"{extract_name.title()} extract: {extract_path.name}")
    display(preview)

## Results

### 4. Build the Canonical Weekly Table

In [12]:
modelling_table, treatment_log = prepare_modelling_table(extract_paths)

display(modelling_table.head(5))
display(modelling_table.tail(5))
display(treatment_log)

,date,sales,holiday,sales_promotion,competitor_spend,instagram_spend,google_ads_spend,tv_spend,youtube_spend,newspaper_spend,influencer_spend,ott_spend,instagram_spend_was_missing,google_ads_spend_was_missing,tv_spend_was_missing,youtube_spend_was_missing,newspaper_spend_was_missing,influencer_spend_was_missing,ott_spend_was_missing,competitor_spend_was_missing
0,2022-01-03,1084031.0,0.0,Coupons,237000.0,0.0,94850.0,47580.0,45840.0,0.0,0.0,73570.0,True,False,False,False,True,True,False,False
1,2022-01-10,1225796.0,NaN,NaN,299300.0,0.0,0.0,60140.0,0.0,0.0,0.0,0.0,True,True,False,True,True,True,True,True
2,2022-01-17,1445628.0,0.0,Flash Sale,361600.0,90520.0,0.0,52180.0,30710.0,0.0,48300.0,0.0,False,True,False,False,True,False,True,False
3,2022-01-24,1445965.0,0.0,Normal,293650.0,77800.0,0.0,41940.0,39610.0,0.0,49160.0,0.0,False,True,False,False,True,False,True,False
4,2022-01-31,1370267.0,0.0,Normal,681600.0,180380.0,69560.0,29720.0,84310.0,0.0,54920.0,0.0,False,False,False,False,True,False,True,False


,date,sales,holiday,sales_promotion,competitor_spend,instagram_spend,google_ads_spend,tv_spend,youtube_spend,newspaper_spend,influencer_spend,ott_spend,instagram_spend_was_missing,google_ads_spend_was_missing,tv_spend_was_missing,youtube_spend_was_missing,newspaper_spend_was_missing,influencer_spend_was_missing,ott_spend_was_missing,competitor_spend_was_missing
204,2025-12-01,2824619.0,0.0,Normal,292950.0,79920.0,110120.0,32600.0,22270.0,7200.0,0.0,38990.0,False,False,False,False,False,True,False,False
205,2025-12-08,2747119.0,NaN,NaN,292250.0,0.0,0.0,70010.0,0.0,6970.0,0.0,0.0,True,True,False,True,False,True,True,True
206,2025-12-15,2676494.0,0.0,Coupons,291550.0,101390.0,0.0,62690.0,26080.0,6700.0,30940.0,77400.0,False,True,False,False,False,False,False,False
207,2025-12-22,2489367.0,0.0,Normal,265650.0,69870.0,0.0,46370.0,53250.0,9370.0,35660.0,52510.0,False,True,False,False,False,False,False,False
208,2025-12-29,2299871.0,0.0,Normal,354750.0,65320.0,0.0,58950.0,38760.0,5350.0,35010.0,56360.0,False,True,False,False,False,False,False,False


,column,affected_rows,treatment
0,instagram_spend,68,Filled with zero as an inactive media week
1,google_ads_spend,94,Filled with zero as an inactive media week
2,tv_spend,57,Filled with zero as an inactive media week
3,youtube_spend,80,Filled with zero as an inactive media week
4,newspaper_spend,64,Filled with zero as an inactive media week
5,influencer_spend,79,Filled with zero as an inactive media week
6,ott_spend,71,Filled with zero as an inactive media week
7,competitor_spend,38,Interpolated in time order; boundary values fo...
8,sales,3,Retained missing values; exclude from model fi...


### 5. Validate the Weekly Contract

In [13]:
modelling_dates = pd.to_datetime(modelling_table["date"], errors="raise")
expected_dates = pd.date_range(
    modelling_dates.min(), modelling_dates.max(), freq="W-MON"
)

assert len(modelling_table) == 209, "Expected exactly 209 weekly rows"
assert modelling_dates.is_unique, "Weekly dates must be unique"
assert modelling_dates.is_monotonic_increasing, "Rows must be date-sorted"
assert modelling_dates.tolist() == expected_dates.tolist(), (
    "The modelling table must contain every Monday in the date range"
)
assert int(modelling_table["sales"].isna().sum()) == 3

validation_summary = pd.DataFrame(
    {
        "check": [
            "weekly rows",
            "unique weekly dates",
            "continuous Monday calendar",
            "retained missing sales rows",
        ],
        "observed": [
            len(modelling_table),
            modelling_dates.nunique(),
            len(expected_dates),
            int(modelling_table["sales"].isna().sum()),
        ],
    }
)
display(validation_summary)

,check,observed
0,weekly rows,209
1,unique weekly dates,209
2,continuous Monday calendar,209
3,retained missing sales rows,3


### 6. Export Prepared Data and the Treatment Log

In [14]:
modelling_output = paths.processed / "modelling_ready_data.csv"
treatment_output = paths.reports / "data_treatment_log.csv"

modelling_table.to_csv(modelling_output, index=False)
treatment_log.to_csv(treatment_output, index=False)

print(f"Saved {modelling_output.relative_to(PROJECT_ROOT)}")
print(f"Saved {treatment_output.relative_to(PROJECT_ROOT)}")

Saved data\processed\modelling_ready_data.csv
Saved output\reports\data_treatment_log.csv


## Takeaways

- The four input formats reconcile to 209 unique, continuous Monday weeks.
- The treatment log makes the synthetic missing-value assumptions auditable rather than hiding them.
- Three missing sales observations remain visible and will be excluded only when model fitting requires an observed target.